### Project: Analysis of MRI Data of Patients with Multiple Sclerosis
### Author: Ilya Glukhov
### Description: ETL pipeline, EDA, visualization, and A/B testing on MosMedData SCLEROSIS dataset

Import libraries, style configuration, implementation of helper functions

In [1]:
import numpy as np
import pandas as pd
import pydicom
import sqlite3
import os
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu, shapiro

# 2. GLOBAL SETTINGS
# Matplotlib style settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style("whitegrid")

# Path settings
NOTEBOOK_DIR = Path.cwd()
# Dataset is located in the same folder as the notebook
DATASET_PATH = NOTEBOOK_DIR / "SCLEROSIS21_172/SCLEROSIS21_172"
OUTPUT_DIR = NOTEBOOK_DIR 

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Color palette
COLORS = {
    'primary': '#3B82F6',
    'secondary': '#EF4444',
    'accent1': '#10B981',
    'accent2': '#F59E0B',
    'accent3': '#8B5CF6',
    'accent4': '#EC4899',
    'ms': {0: '#3B82F6', 1: '#EF4444'}
}


# 3. HELPER FUNCTIONS
def find_patient_folder(study_uid):
    """Finds the patient folder path by study_uid"""
    for folder in DATASET_PATH.iterdir():
        if folder.is_dir() and not folder.name.startswith('.'):
            study_folder = folder / study_uid
            if study_folder.exists():
                return study_folder, folder.name
    return None, None


def extract_dicom_metadata(dicom_path):
    """Extracts metadata from a DICOM file (without pixels)"""
    try:
        dcm = pydicom.dcmread(dicom_path, stop_before_pixels=True)
        return {
            'manufacturer': dcm.get('Manufacturer', None),
            'manufacturer_model': dcm.get('ManufacturerModelName', None),
            'echo_time': float(dcm.get('EchoTime', np.nan)) if dcm.get('EchoTime') else np.nan,
            'repetition_time': float(dcm.get('RepetitionTime', np.nan)) if dcm.get('RepetitionTime') else np.nan,
            'slice_thickness': float(dcm.get('SliceThickness', np.nan)) if dcm.get('SliceThickness') else np.nan,
            'flip_angle': float(dcm.get('FlipAngle', np.nan)) if dcm.get('FlipAngle') else np.nan,
            'modality': dcm.get('Modality', None),
        }
    except Exception:
        return None

**ETL** pipeline

In [2]:
# 4. ETL PIPELINE
print("ETL PIPELINE: MosMedData SCLEROSIS")
print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# 4.1 EXTRACT
print("\nSTEP 1: EXTRACT")

labels_path = DATASET_PATH / 'labels.xlsx'
df_labels = pd.read_excel(labels_path)
df_labels['study_uid'] = df_labels['study_uid'].astype(str)

print(f"labels.xlsx loaded: {len(df_labels)} records, distribution: {df_labels['sclerosis'].value_counts().to_dict()}")

print("Collecting metadata from DICOM files...")
records = []
error_details = []

for idx, row in df_labels.iterrows():
    study_uid = row['study_uid']
    sclerosis = row['sclerosis']
    
    print(f"[{idx+1:3d}/172] {study_uid[:40]}...", end=" ")
    
    study_folder, source_folder = find_patient_folder(study_uid)
    
    if not study_folder:
        error_details.append({'study_uid': study_uid, 'reason': 'folder_not_found'})
        print("(folder not found)")
        continue
    
    series_folders = [f for f in study_folder.iterdir() if f.is_dir()]
    if not series_folders:
        error_details.append({'study_uid': study_uid, 'reason': 'no_series_folders'})
        print("(no series)")
        continue
    
    first_series = series_folders[0]
    dcm_files = list(first_series.glob("*.dcm"))
    
    if not dcm_files:
        error_details.append({'study_uid': study_uid, 'reason': 'no_dicom_files'})
        print("(no DICOM files)")
        continue
    
    metadata = extract_dicom_metadata(dcm_files[0])
    if not metadata:
        error_details.append({'study_uid': study_uid, 'reason': 'metadata_extraction_failed'})
        print("(metadata extraction failed)")
        continue
    
    record = {
        'study_uid': study_uid,
        'sclerosis': sclerosis,
        'source_folder': source_folder,
        'num_series': len(series_folders),
        'num_slices': len(dcm_files),
        **metadata
    }
    records.append(record)
    print("success")

print(f"Extract results: successful {len(records)}/{len(df_labels)}, errors: {len(error_details)}")

df_raw = pd.DataFrame(records)
raw_csv_path = OUTPUT_DIR / "raw_extracted_data.csv"
df_raw.to_csv(raw_csv_path, index=False)
print(f"Raw data saved: {raw_csv_path}")

# 4.2 TRANSFORM
print("\nSTEP 2: TRANSFORM")

df = df_raw.copy()

# Check for missing values
null_counts = df.isnull().sum()
null_percent = (null_counts / len(df)) * 100
print("Missing values:")
for col in null_counts[null_counts > 0].index:
    print(f"  {col}: {null_counts[col]} ({null_percent[col]:.2f}%)")

# Clean string columns
string_columns = ['manufacturer', 'manufacturer_model', 'source_folder']
for col in string_columns:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.upper()
        df[col] = df[col].replace(['NAN', 'NONE', ''], np.nan)

# Group manufacturers
if 'manufacturer' in df.columns:
    df['manufacturer_clean'] = df['manufacturer'].apply(
        lambda x: 'TOSHIBA' if x in ['TOSHIBA_MEC', 'TOSHIBA'] else x
    )
    df['manufacturer_group'] = df['manufacturer_clean'].apply(
        lambda x: 'TOSHIBA' if x == 'TOSHIBA' else 'OTHER'
    )
    print("Manufacturer distribution after grouping:")
    print(df['manufacturer_group'].value_counts().to_string())

# Categorize Echo Time
if 'echo_time' in df.columns:
    df['te_category'] = pd.cut(
        df['echo_time'],
        bins=[0, 60, 100, 200],
        labels=['short (<60ms)', 'medium (60-100ms)', 'long (>100ms)']
    )
    print("Echo Time category distribution:")
    print(df['te_category'].value_counts().to_string())

# Outlier analysis
numeric_cols = ['echo_time', 'repetition_time', 'slice_thickness', 'flip_angle', 'num_series', 'num_slices']
numeric_cols = [col for col in numeric_cols if col in df.columns]

for col in numeric_cols:
    data = df[col].dropna()
    if len(data) > 0:
        Q1 = data.quantile(0.25)
        Q3 = data.quantile(0.75)
        IQR = Q3 - Q1
        outliers = data[(data < Q1 - 1.5 * IQR) | (data > Q3 + 1.5 * IQR)]
        print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(data)*100:.1f}%)")

# Statistics by group
for col in numeric_cols:
    if col in df.columns:
        stats = df.groupby('sclerosis')[col].agg(['mean', 'median'])
        if not stats.isnull().all().all():
            print(f"\n{col}:")
            print(f"  MS (1):     mean={stats.loc[1, 'mean']:.2f}, median={stats.loc[1, 'median']:.2f}")
            print(f"  Normal (0): mean={stats.loc[0, 'mean']:.2f}, median={stats.loc[0, 'median']:.2f}")

# Final cleaning
df = df.drop_duplicates(subset=['study_uid'])
column_order = ['study_uid', 'sclerosis', 'source_folder', 'manufacturer', 'manufacturer_clean', 
                'manufacturer_group', 'manufacturer_model', 'echo_time', 'te_category', 
                'repetition_time', 'slice_thickness', 'flip_angle', 'num_series', 'num_slices', 'modality']
existing_cols = [col for col in column_order if col in df.columns]
df = df[existing_cols + [col for col in df.columns if col not in existing_cols]]

print(f"Final size: {df.shape[0]} rows x {df.shape[1]} columns")

# 4.3 LOAD
print("\nSTEP 3: LOAD")

csv_path = OUTPUT_DIR / "cleaned_data.csv"
df.to_csv(csv_path, index=False)
print(f"CSV saved: {csv_path} ({csv_path.stat().st_size / 1024:.1f} KB)")

db_path = OUTPUT_DIR / "sclerosis.db"
conn = sqlite3.connect(db_path)
df.to_sql('patients', conn, if_exists='replace', index=False)
manufacturers_df = df[['manufacturer', 'manufacturer_group']].drop_duplicates().dropna()
manufacturers_df.to_sql('manufacturers', conn, if_exists='replace', index=False)
conn.close()
print(f"SQLite saved: {db_path} (tables: patients, manufacturers)")

print(f"Final statistics: {len(df)} patients, MS: {(df['sclerosis']==1).sum()}, Normal: {(df['sclerosis']==0).sum()}")

ETL PIPELINE: MosMedData SCLEROSIS
Start: 2026-05-12 22:30:09

STEP 1: EXTRACT
labels.xlsx loaded: 172 records, distribution: {0: 86, 1: 86}
[  1/172] 1.2.643.5.1.13.13.12.2.77.8252.011411090... success
[  2/172] 1.2.643.5.1.13.13.12.2.77.8252.131500041... success
[  3/172] 1.2.643.5.1.13.13.12.2.77.8252.010305060... success
[  4/172] 1.2.643.5.1.13.13.12.2.77.8252.071406031... success
[  5/172] 1.2.643.5.1.13.13.12.2.77.8252.121511041... success
[  6/172] 1.2.643.5.1.13.13.12.2.77.8252.080607010... success
[  7/172] 1.2.643.5.1.13.13.12.2.77.8252.061311151... success
[  8/172] 1.2.643.5.1.13.13.12.2.77.8252.120308041... success
[  9/172] 1.2.643.5.1.13.13.12.2.77.8252.010902121... success
[ 10/172] 1.2.643.5.1.13.13.12.2.77.8252.070702060... success
[ 11/172] 1.2.643.5.1.13.13.12.2.77.8252.070313041... success
[ 12/172] 1.2.643.5.1.13.13.12.2.77.8252.141213090... success
[ 13/172] 1.2.643.5.1.13.13.12.2.77.8252.051014080... success
[ 14/172] 1.2.643.5.1.13.13.12.2.77.8252.081310040...

Visualization

In [3]:
# 5. VISUALIZATION
print("\nVISUALIZATION")

plots_dir = OUTPUT_DIR / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

# Plot 1: Scanner manufacturers
manufacturer_counts = df['manufacturer_group'].value_counts()
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(manufacturer_counts.index, manufacturer_counts.values, 
               color=[COLORS['primary'], COLORS['accent3']], edgecolor='black', linewidth=1)

for bar, val in zip(bars, manufacturer_counts.values):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2, 
            f'{val} ({val/len(df)*100:.1f}%)', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Number of patients')
ax.set_title('Distribution of patients by scanner manufacturer')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(plots_dir / '01_manufacturer_bar.png', dpi=300, bbox_inches='tight')
plt.close()

# Plot 2: Echo Time categories
te_order = ['short (<60ms)', 'medium (60-100ms)', 'long (>100ms)']
te_counts = df['te_category'].value_counts()[te_order]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(te_counts.index, te_counts.values, 
              color=[COLORS['accent1'], COLORS['accent2'], COLORS['primary']], 
              edgecolor='black', linewidth=1.5)

for bar, val in zip(bars, te_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
            f'{val}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xlabel('Echo Time (TE) category')
ax.set_ylabel('Number of patients')
ax.set_title('Distribution of patients by Echo Time (TE) duration')
ax.set_ylim(0, max(te_counts.values) + 15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(plots_dir / '02_te_category_bar.png', dpi=300, bbox_inches='tight')
plt.close()

# Plot 3: Echo Time categories vs MS
crosstab = pd.crosstab(df['te_category'], df['sclerosis'], normalize='index') * 100
crosstab = crosstab.reindex(te_order)
abs_crosstab = pd.crosstab(df['te_category'], df['sclerosis']).reindex(te_order)

fig, ax = plt.subplots(figsize=(10, 6))
bottom = np.zeros(len(crosstab))

for sclerosis in [0, 1]:
    values = crosstab[sclerosis].values
    label = 'Normal (0)' if sclerosis == 0 else 'MS (1)'
    color = COLORS['ms'][sclerosis]
    
    bars = ax.bar(crosstab.index, values, bottom=bottom, 
                  label=label, color=color, edgecolor='black', linewidth=0.5)
    
    for bar, val, abs_val in zip(bars, values, abs_crosstab[sclerosis].values):
        if val > 8:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_y() + bar.get_height()/2,
                    f'{val:.0f}%\n(n={abs_val})', ha='center', va='center', 
                    fontsize=9, color='white', fontweight='bold')
    bottom += values

total_patients = abs_crosstab.sum(axis=1)
for i, total in enumerate(total_patients):
    ax.text(i, 102, f'n = {total} patients', ha='center', fontsize=10, fontweight='bold', color='gray')

ax.set_xlabel('Echo Time (TE) category')
ax.set_ylabel('Percentage of patients (%)')
ax.set_title('Proportion of patients with multiple sclerosis by Echo Time category')
ax.set_ylim(0, 105)
ax.legend(loc='upper right', title='Diagnosis')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(plots_dir / '03_te_category_vs_ms_stacked.png', dpi=300, bbox_inches='tight')
plt.close()

print(f"Plots saved to {plots_dir}")


VISUALIZATION
Plots saved to /Users/ilya_gluhov/Desktop/Projects/Project MMD/plots


**A/B Test**

In [4]:
# 6. A/B TEST: number of series
print("\nA/B TEST: Number of series (num_series) vs Diagnosis (sclerosis)")

ms_series = df[df['sclerosis'] == 1]['num_series'].dropna()
normal_series = df[df['sclerosis'] == 0]['num_series'].dropna()

print(f"Descriptive statistics:")
print(f"  MS: n={len(ms_series)}, mean={ms_series.mean():.2f}, median={ms_series.median():.0f}")
print(f"  Normal: n={len(normal_series)}, mean={normal_series.mean():.2f}, median={normal_series.median():.0f}")
print(f"  Difference: mean = {ms_series.mean() - normal_series.mean():.2f}, median = {ms_series.median() - normal_series.median():.0f}")

# Normality check
_, p_ms = shapiro(ms_series)
_, p_norm = shapiro(normal_series)
print(f"Shapiro-Wilk p-value: MS = {p_ms:.4f}, Normal = {p_norm:.4f}")
if p_ms < 0.05 or p_norm < 0.05:
    print("Distributions are NOT normal -> using Mann-Whitney U test")
else:
    print("Distributions are normal -> can use T-test")

# Mann-Whitney U test
u_stat, p_value = mannwhitneyu(ms_series, normal_series, alternative='two-sided')
print(f"Mann-Whitney U test: U = {u_stat:.2f}, p-value = {p_value:.6f}")

if p_value < 0.05:
    print("Result: STATISTICALLY SIGNIFICANT (p < 0.05)")
    print(f"  MS patients undergo on average 1 more series (median: MS={ms_series.median():.0f}, Normal={normal_series.median():.0f})")
else:
    print("Result: NOT SIGNIFICANT (p >= 0.05)")

# A/B test visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Boxplot
data_to_plot = [normal_series.values, ms_series.values]
box = axes[0].boxplot(data_to_plot, tick_labels=['Normal (0)', 'MS (1)'],
                       patch_artist=True, widths=0.6,
                       medianprops=dict(linewidth=2, color='black'),
                       flierprops=dict(marker='o', markerfacecolor='gray', markersize=6, alpha=0.5))
box['boxes'][0].set_facecolor(COLORS['ms'][0])
box['boxes'][1].set_facecolor(COLORS['ms'][1])
box['boxes'][0].set_alpha(0.7)
box['boxes'][1].set_alpha(0.7)

axes[0].set_ylabel('Number of series')
axes[0].set_title('Boxplot: Number of series by group')
axes[0].grid(axis='y', alpha=0.3)
axes[0].text(1, normal_series.median() + 0.3, f'median={normal_series.median():.0f}', 
             ha='center', fontsize=10, color=COLORS['ms'][0], fontweight='bold')
axes[0].text(2, ms_series.median() + 0.3, f'median={ms_series.median():.0f}', 
             ha='center', fontsize=10, color=COLORS['ms'][1], fontweight='bold')

# Violin plot
df_violin = df[['num_series', 'sclerosis']].dropna().copy()
df_violin['sclerosis'] = df_violin['sclerosis'].map({0: 'Normal (0)', 1: 'MS (1)'})

# Fix: palette dictionary keys must match values in df_violin['sclerosis']
sns.violinplot(data=df_violin, x='sclerosis', y='num_series',
               hue='sclerosis', legend=False,
               palette={'Normal (0)': COLORS['ms'][0], 'MS (1)': COLORS['ms'][1]},
               ax=axes[1], inner='quartile', linewidth=1.5)
axes[1].set_xlabel('Diagnosis')
axes[1].set_ylabel('Number of series')
axes[1].set_title('Violin plot: Distribution density')
axes[1].grid(axis='y', alpha=0.3)

# Histogram
bins = range(int(min(df['num_series'].min(), normal_series.min(), ms_series.min())), 
             int(max(df['num_series'].max(), normal_series.max(), ms_series.max())) + 2)
axes[2].hist(normal_series, bins=bins, alpha=0.6, label='Normal (0)', 
             color=COLORS['ms'][0], edgecolor='black', linewidth=1)
axes[2].hist(ms_series, bins=bins, alpha=0.6, label='MS (1)', 
             color=COLORS['ms'][1], edgecolor='black', linewidth=1)
axes[2].set_xlabel('Number of series')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Histogram of distribution')
axes[2].legend()
axes[2].grid(axis='y', alpha=0.3)

for ax in axes:
    ax.text(0.98, 0.97, f'p-value = {p_value:.4f}', transform=ax.transAxes, 
            ha='right', va='top', fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(plots_dir / 'ab_test_num_series.png', dpi=300, bbox_inches='tight')
plt.close()

print(f"A/B test plot saved: {plots_dir / 'ab_test_num_series.png'}")


A/B TEST: Number of series (num_series) vs Diagnosis (sclerosis)
Descriptive statistics:
  MS: n=85, mean=11.13, median=11
  Normal: n=86, mean=10.10, median=10
  Difference: mean = 1.02, median = 1
Shapiro-Wilk p-value: MS = 0.0000, Normal = 0.0000
Distributions are NOT normal -> using Mann-Whitney U test
Mann-Whitney U test: U = 4333.50, p-value = 0.034551
Result: STATISTICALLY SIGNIFICANT (p < 0.05)
  MS patients undergo on average 1 more series (median: MS=11, Normal=10)
A/B test plot saved: /Users/ilya_gluhov/Desktop/Projects/Project MMD/plots/ab_test_num_series.png


In [1]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.
